# Лабораторная работа
**Тема:** Среда Python и Jupyter Notebook  
**Акцент:** Структура программы, данные, проверки, обработка ошибок, отладка, среда выполнения  
**Задача:** Реализовать модуль статистического анализа числовых данных (среднее, медиана, мода).

## 1. Исходные данные и требования
- На вход подаётся список чисел (целых, вещественных, строковые представления, `None`).
- Нужно корректно обрабатывать пустые списки, один элемент, повторяющиеся значения, нечисловые данные.
- Программа должна быть структурирована (отдельные функции), использовать логирование, обработку ошибок и быть воспроизводимой в Jupyter.

In [ ]:
import logging
from collections import Counter
from typing import List, Union

# Настройка логирования (отладка)
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s:%(message)s')
logger = logging.getLogger(__name__)

In [ ]:
def clean_data(data: List) -> List[Union[int, float]]:
    """Очистка данных: удаление None, преобразование строк в числа, проверка типов."""
    cleaned = []
    for idx, val in enumerate(data):
        if val is None:
            logger.warning(f"Пропущен None на позиции {idx}")
            continue
        try:
            if isinstance(val, (int, float)):
                cleaned.append(val)
            elif isinstance(val, str):
                num = float(val.replace(',', '.'))
                cleaned.append(num)
            else:
                raise TypeError(f"Неподдерживаемый тип: {type(val)}")
        except (ValueError, TypeError) as e:
            logger.error(f"Ошибка преобразования на индексе {idx}: {val} -> {e}")
            raise ValueError(f"Некорректное значение: {val}") from e
    logger.debug(f"Очищенные данные: {cleaned}")
    return cleaned

In [ ]:
def arithmetic_mean(data: List[Union[int, float]]) -> float:
    if not data:
        raise ValueError("Список пуст, среднее не определено")
    return sum(data) / len(data)

def median(data: List[Union[int, float]]) -> float:
    if not data:
        raise ValueError("Список пуст, медиана не определена")
    sorted_data = sorted(data)
    n = len(sorted_data)
    mid = n // 2
    if n % 2 == 0:
        return (sorted_data[mid-1] + sorted_data[mid]) / 2
    else:
        return sorted_data[mid]

def mode(data: List[Union[int, float]]) -> List[Union[int, float]]:
    if not data:
        raise ValueError("Список пуст, мода не определена")
    counter = Counter(data)
    max_freq = max(counter.values())
    modes = [val for val, freq in counter.items() if freq == max_freq]
    logger.debug(f"Найдены моды: {modes} с частотой {max_freq}")
    return modes

In [ ]:
def analyze(data: List) -> dict:
    """Главная функция: очистка + вычисление статистик."""
    try:
        cleaned = clean_data(data)
        if not cleaned:
            raise ValueError("После очистки не осталось числовых данных")
        return {
            "mean": arithmetic_mean(cleaned),
            "median": median(cleaned),
            "mode": mode(cleaned)
        }
    except Exception as e:
        logger.critical(f"Анализ не удался: {e}")
        return {"error": str(e)}

## 2. Тестовые данные (граничные случаи)

In [ ]:
test_cases = [
    [1, 2, 3, 4, 5],                     # обычный
    [1, 1, 2, 2, 3],                     # несколько мод
    [10],                                # один элемент
    [],                                  # пустой
    [1, None, 3, "4.5", "5,2"],         # смешанные типы
    ["abc", 2, 3],                       # нечисловая строка
    None                                 # неверный тип входа
]

## 3. Запуск проверок на пограничных случаях

In [ ]:
def run_tests():
    for i, case in enumerate(test_cases, 1):
        print(f"\n--- Тест {i}: {case} ---")
        result = analyze(case)
        print("Результат:", result)

run_tests()

## 4. Дополнительные модульные проверки (assert)

In [ ]:
def test_mean():
    assert arithmetic_mean([2,4,6]) == 4.0
    try:
        arithmetic_mean([])
    except ValueError:
        pass
    print("Тест среднего пройден")

test_mean()

## 5. Рефлексия
**Основной риск:** некорректная обработка входных данных (None, строки, пустые списки).  
**Как закрыт:**
- Функция `clean_data` нормализует и логирует все проблемы.
- Каждая статистическая функция проверяет список на пустоту.
- Главная функция `analyze` перехватывает любые исключения и возвращает безопасный словарь с ошибкой.
- Добавлены тесты граничных случаев и отладочный вывод (`logger.debug`).

**Итог:** Решение полностью воспроизводимо в Jupyter Notebook, соответствует критериям структуры, данных, проверок, обработки ошибок и отладки.